# Linear classifier training


## 1. Locate the script and the input data

In [ ]:
import shutil
from pathlib import Path

WORK = Path("/kaggle/working/head_clf_stratified")
WORK.mkdir(parents=True, exist_ok=True)

hits = sorted(Path("/kaggle/input").rglob("run_kfold_head.py"))
assert hits, (
    "run_kfold_head.py not found under /kaggle/input -- attach the freshly uploaded "
    "head_clf_stratified bundle or full repo as a Dataset."
)
bundle = hits[0].parent

required_files = (
    "train_head.py",
    "run_kfold_head.py",
    "utils.py",
    "config.yaml",
    "requirements.txt",
)
for name in required_files:
    src = bundle / name
    assert src.exists(), f"{name} missing from attached bundle -- re-upload the Kaggle Dataset."
    shutil.copy(src, WORK / name)

required_data = (
    "train.parquet",
    "test.parquet",
    "train_node_ids.txt",
    "test_node_ids.txt",
    "fold_assignment.csv",
    "published_head_config.json",
    "contrast_summary_header.json",
)

def missing_data(data_dir: Path) -> list[str]:
    return [name for name in required_data if not (data_dir / name).exists()]

def find_data_dir(bundle_dir: Path) -> Path | None:
    candidates = (
        bundle_dir.parent / "data" / "fold_data",
        bundle_dir / "data",
    )
    for candidate in candidates:
        if candidate.exists() and not missing_data(candidate):
            return candidate
    for train_path in sorted(Path("/kaggle/input").rglob("fold_data/train.parquet")):
        candidate = train_path.parent
        if not missing_data(candidate):
            return candidate
    return None

src_data = find_data_dir(bundle)
assert src_data is not None, (
    "fixed fold data not found -- attach a full repo Dataset with data/fold_data/ "
    "or stage kaggle_classifier_training/data/ from data/fold_data/."
)
shutil.copytree(src_data, WORK / "data", dirs_exist_ok=True)

print("bundle      :", bundle)
print("data source :", src_data)
print("work        :", WORK)
print("data        :", WORK / "data")


## 2. Dependencies

Kaggle already ships torch, pandas, numpy and tqdm.

In [ ]:
!pip -q install -r /kaggle/working/head_clf_stratified/requirements.txt

## 3. Check the GPU is actually on

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU -- set Settings > Accelerator to GPU T4/P100."
props = torch.cuda.get_device_properties(0)
print(f"{props.name}, {props.total_memory / 2**30:.1f} GB")
print("config is locked to the published head run; do not edit batch_size for this experiment.")


## 4. Config

Model/head tunables live in `config.yaml`; fixed data/preprocessing constants stay in code. `run_kfold_head.py` compares the loaded run contract against `data/published_head_config.json` before training and fails on any metric-affecting mismatch.


In [ ]:
print((WORK / "config.yaml").read_text())

## 5. Validate the fixed data and fold assignment

The pre-generated files in `data/` are the experiment inputs. This cell checks their schema, label counts, row-aligned node-id sidecars, and the copied fold assignment; it does not regenerate the split.


In [ ]:
import json
import pandas as pd

expected = {
    "train": {"rows": 6077, "unsafe": 639, "safe": 5438},
    "test": {"rows": 2603, "unsafe": 273, "safe": 2330},
}
frames = {}
ids_by_split = {}
for split, exp in expected.items():
    df = pd.read_parquet(WORK / "data" / f"{split}.parquet")
    ids = (WORK / "data" / f"{split}_node_ids.txt").read_text().splitlines()
    assert list(df.columns) == ["text", "label"], (split, df.columns)
    assert len(df) == exp["rows"], (split, len(df), exp["rows"])
    assert len(ids) == len(df), (split, len(ids), len(df))
    assert len(set(ids)) == len(ids), f"duplicate node IDs in {split}"
    unsafe = int(df["label"].sum())
    safe = int((1 - df["label"]).sum())
    assert (unsafe, safe) == (exp["unsafe"], exp["safe"]), (split, unsafe, safe)
    assert df["text"].notna().all()
    assert df["text"].str.strip().ne("").all()
    frames[split] = df
    ids_by_split[split] = ids
    print(f"{split:5s}: {len(df):5d} rows | {unsafe:4d} unsafe / {safe:4d} safe")

assert not (set(ids_by_split["train"]) & set(ids_by_split["test"])), "train/test node IDs overlap"
folds = pd.read_csv(WORK / "data" / "fold_assignment.csv", dtype={"node_id": str, "severity": str, "fold": int})
assert list(folds.columns) == ["node_id", "severity", "fold"], folds.columns
assert len(folds) == len(ids_by_split["train"])
assert folds["node_id"].nunique() == len(folds)
assert set(folds["node_id"]) == set(ids_by_split["train"])
assert sorted(folds["fold"].unique().tolist()) == [0, 1, 2, 3, 4]

aligned = pd.DataFrame({"node_id": ids_by_split["train"], "label": frames["train"]["label"].to_numpy()}).merge(
    folds, on="node_id", how="left", validate="one_to_one"
)
labels_from_severity = aligned["severity"].astype(str).isin({"3", "4", "5"}).astype("int64")
assert (labels_from_severity.to_numpy() == aligned["label"].to_numpy()).all()

published = json.loads((WORK / "data" / "published_head_config.json").read_text())
assert published["max_epochs"] == 15 and published["checkpoint_every"] == 2
assert json.loads((WORK / "data" / "contrast_summary_header.json").read_text())[0] == "fold"

print(f"fold assignment by severity:")
display(folds.groupby(["fold", "severity"]).size().unstack(fill_value=0))
display(frames["train"].head(3))


## 6. Inspect the model

This loads the frozen encoder and classifier head in a short subprocess, prints the module tree,
and exits so GPU memory is released before training.

In [ ]:
import subprocess
import sys
import textwrap

code = r"""
from train_head import Config, SafetyClassifier, DEFAULT_CONFIG

cfg = Config.load(DEFAULT_CONFIG)
model = SafetyClassifier(cfg)
print(model)
print('\nHEAD ONLY')
print(model.head)
"""

subprocess.run([sys.executable, "-c", textwrap.dedent(code)], cwd=WORK, check=True)

## 7. Run the 5-fold experiment

Outputs land in `/kaggle/working/head_clf_stratified/runs_kfold/noprefix_h0/`. The runner validates the config, data contract, and fold assignment before training.


In [ ]:
!cd /kaggle/working/head_clf_stratified && python run_kfold_head.py --model-config-name noprefix_h0


## 8. Results

One row per fold, followed by `mean` and sample `std` (`ddof=1`). The summary uses the same columns as the contrast k-fold summary.


In [ ]:
import json
import pandas as pd

run = WORK / "runs_kfold" / "noprefix_h0"
summary = pd.read_csv(run / "summary.csv")
manifest = json.loads((run / "kfold_manifest.json").read_text())
print(run)
print("target_recall:", manifest["target_recall"])
print("total hours:", round(manifest["total_wall_seconds"] / 3600, 2))

cols = [
    "fold", "threshold", "validation_recall",
    "tuned_n_flagged", "tuned_flag_rate", "tuned_precision", "tuned_recall",
    "tuned_average_precision", "tuned_roc_auc",
]
pd.set_option("display.width", 250)
display(summary[cols].round(4))

mean = summary.loc[summary["fold"] == "mean"].iloc[0]
std = summary.loc[summary["fold"] == "std"].iloc[0]
print(
    f"Average Precision: {mean['tuned_average_precision']:.4f} +/- "
    f"{std['tuned_average_precision']:.4f}"
)
print(f"ROC-AUC: {mean['tuned_roc_auc']:.4f} +/- {std['tuned_roc_auc']:.4f}")
print(
    f"Precision @ recall .80: {mean['tuned_precision']:.4f} +/- "
    f"{std['tuned_precision']:.4f}"
)
print(f"Forwarding rate: {mean['tuned_flag_rate']:.4f} +/- {std['tuned_flag_rate']:.4f}")


## 9. Eyeball and package outputs

The score files are row-aligned with `data/test.parquet`. The zip is written under `/kaggle/working` for download from the notebook output panel.


In [ ]:
import shutil

run = WORK / "runs_kfold" / "noprefix_h0"
summary = pd.read_csv(run / "summary.csv")
print("final summary:")
display(summary[[
    "fold", "tuned_precision", "tuned_recall",
    "tuned_average_precision", "tuned_roc_auc",
]].round(4))

fold = 0
scores = pd.read_parquet(run / f"fold_{fold}" / "test_scores.parquet")
scores["text"] = pd.read_parquet(WORK / "data" / "test.parquet")["text"].values
print(f"
top 10 by P(unsafe), fold {fold}
")
for _, r in scores.sort_values("score", ascending=False).head(10).iterrows():
    print(f"[tuned={bool(r.flag_tuned)} P(unsafe)={r.score:.3f}]  {r.text[:200]}
")

archive = shutil.make_archive("/kaggle/working/head_clf_stratified_kfold_results", "zip", run)
print("
zip:", archive)
